# Lasso Regression Analysis

Apply Lasso regression to select significant variables, then re-fit OLS with the reduced feature set.

## Setup

Load libraries and data.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# Input files
IN_SAMPLE_FILE = Path("Oil_InSample.csv")
OUT_SAMPLE_FILE = Path("Oil_OutOfSample.csv")

## Load Data

In [3]:
# Load datasets
df_insample = pd.read_csv(IN_SAMPLE_FILE, index_col=0, parse_dates=True)
df_outsample = pd.read_csv(OUT_SAMPLE_FILE, index_col=0, parse_dates=True)

print("In-Sample Data:")
print(f"  Shape: {df_insample.shape}")
print(f"  Date range: {df_insample.index.min().date()} → {df_insample.index.max().date()}")

print("\nOut-of-Sample Data:")
print(f"  Shape: {df_outsample.shape}")
print(f"  Date range: {df_outsample.index.min().date()} → {df_outsample.index.max().date()}")

In-Sample Data:
  Shape: (1195, 26)
  Date range: 1995-02-10 → 2017-12-29

Out-of-Sample Data:
  Shape: (339, 26)
  Date range: 2018-01-05 → 2024-06-28


## Prepare Data

In [4]:
# Extract target and features
y_insample = df_insample['cl1_ret'].values
X_insample = df_insample.drop('cl1_ret', axis=1).values
X_insample_cols = df_insample.drop('cl1_ret', axis=1).columns.tolist()

y_outsample = df_outsample['cl1_ret'].values
X_outsample = df_outsample.drop('cl1_ret', axis=1).values

print(f"In-Sample: y shape {y_insample.shape}, X shape {X_insample.shape}")
print(f"Out-of-Sample: y shape {y_outsample.shape}, X shape {X_outsample.shape}")
print(f"\nNumber of features: {len(X_insample_cols)}")

In-Sample: y shape (1195,), X shape (1195, 25)
Out-of-Sample: y shape (339,), X shape (339, 25)

Number of features: 25


## Standardize Features

Lasso requires standardization for fair coefficient comparison.

In [5]:
# Standardize features (crucial for Lasso)
scaler = StandardScaler()
X_insample_scaled = scaler.fit_transform(X_insample)
X_outsample_scaled = scaler.transform(X_outsample)

print("Features standardized:")
print(f"  Mean: {X_insample_scaled.mean(axis=0).round(6).mean():.6f}")
print(f"  Std: {X_insample_scaled.std(axis=0).round(6).mean():.6f}")

Features standardized:
  Mean: 0.000000
  Std: 1.000000


## Apply Lasso with Cross-Validation

Use LassoCV to find the optimal regularization parameter (alpha).

In [6]:
# LassoCV to find optimal alpha
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_insample_scaled, y_insample)

print(f"Optimal alpha (regularization parameter): {lasso_cv.alpha_:.6f}")
print(f"Mean cross-validated MSE: {lasso_cv.mse_path_.mean(axis=1).min():.6f}")

Optimal alpha (regularization parameter): 0.000479
Mean cross-validated MSE: 0.002516


## Fit Lasso Model with Optimal Alpha

In [7]:
# Fit Lasso with optimal alpha
lasso_model = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
lasso_model.fit(X_insample_scaled, y_insample)

# Get predictions
y_pred_lasso_insample = lasso_model.predict(X_insample_scaled)
y_pred_lasso_outsample = lasso_model.predict(X_outsample_scaled)

# Calculate performance metrics
lasso_r2_insample = r2_score(y_insample, y_pred_lasso_insample)
lasso_rmse_insample = np.sqrt(mean_squared_error(y_insample, y_pred_lasso_insample))
lasso_mae_insample = mean_absolute_error(y_insample, y_pred_lasso_insample)

lasso_r2_outsample = r2_score(y_outsample, y_pred_lasso_outsample)
lasso_rmse_outsample = np.sqrt(mean_squared_error(y_outsample, y_pred_lasso_outsample))
lasso_mae_outsample = mean_absolute_error(y_outsample, y_pred_lasso_outsample)

print("="*70)
print("LASSO MODEL PERFORMANCE")
print("="*70)
print("\nIn-Sample:")
print(f"  R² Score:  {lasso_r2_insample:.6f}")
print(f"  RMSE:      {lasso_rmse_insample:.6f}")
print(f"  MAE:       {lasso_mae_insample:.6f}")

print("\nOut-of-Sample:")
print(f"  R² Score:  {lasso_r2_outsample:.6f}")
print(f"  RMSE:      {lasso_rmse_outsample:.6f}")
print(f"  MAE:       {lasso_mae_outsample:.6f}")

LASSO MODEL PERFORMANCE

In-Sample:
  R² Score:  0.058995
  RMSE:      0.048823
  MAE:       0.036032

Out-of-Sample:
  R² Score:  0.001513
  RMSE:      0.061636
  MAE:       0.042430


## Identify Selected Variables

In [8]:
# Get coefficients
lasso_coefs = lasso_model.coef_

# Create coefficient summary
coef_df = pd.DataFrame({
    'Feature': X_insample_cols,
    'Lasso_Coefficient': lasso_coefs,
    'Abs_Coefficient': np.abs(lasso_coefs)
}).sort_values('Abs_Coefficient', ascending=False)

# Identify non-zero coefficients
selected_vars = coef_df[coef_df['Lasso_Coefficient'] != 0].copy()

print(f"\nTotal features: {len(X_insample_cols)}")
print(f"Features selected by Lasso: {len(selected_vars)}")
print(f"Features eliminated by Lasso: {len(X_insample_cols) - len(selected_vars)}")

print("\n" + "="*70)
print("SELECTED VARIABLES (Non-zero coefficients)")
print("="*70)
print(selected_vars.to_string())

# Store selected feature names
selected_feature_names = selected_vars['Feature'].tolist()
print(f"\nSelected features: {selected_feature_names}")


Total features: 25
Features selected by Lasso: 16
Features eliminated by Lasso: 9

SELECTED VARIABLES (Non-zero coefficients)
                              Feature  Lasso_Coefficient  Abs_Coefficient
19   dow_jones_us_oilandgas_index_ret           0.010127         0.010127
20                     exxonmobil_ret          -0.008230         0.008230
13       weekly_commercial_short_diff           0.007829         0.007829
12        weekly_commercial_long_diff          -0.006681         0.006681
11  weekly_change_in_crude_stock_diff          -0.004542         0.004542
5                             cl2_ret          -0.004041         0.004041
1                          cl1_ret_l2          -0.003188         0.003188
7                             spx_ret          -0.002789         0.002789
3                          cl1_ret_l4           0.002332         0.002332
22                        chevron_ret          -0.002027         0.002027
18                           vix_diff          -0.001659   

## All Coefficients (including zeros)

In [9]:
print("\n" + "="*70)
print("ALL LASSO COEFFICIENTS")
print("="*70)
print(coef_df.to_string())


ALL LASSO COEFFICIENTS
                              Feature  Lasso_Coefficient  Abs_Coefficient
19   dow_jones_us_oilandgas_index_ret           0.010127         0.010127
20                     exxonmobil_ret          -0.008230         0.008230
13       weekly_commercial_short_diff           0.007829         0.007829
12        weekly_commercial_long_diff          -0.006681         0.006681
11  weekly_change_in_crude_stock_diff          -0.004542         0.004542
5                             cl2_ret          -0.004041         0.004041
1                          cl1_ret_l2          -0.003188         0.003188
7                             spx_ret          -0.002789         0.002789
3                          cl1_ret_l4           0.002332         0.002332
22                        chevron_ret          -0.002027         0.002027
18                           vix_diff          -0.001659         0.001659
9              weekly_prod_crude_diff           0.001002         0.001002
16            

## Extract Reduced Dataset with Selected Variables

In [10]:
# Create reduced datasets with selected variables
df_insample_reduced = df_insample[['cl1_ret'] + selected_feature_names]
df_outsample_reduced = df_outsample[['cl1_ret'] + selected_feature_names]

print(f"\nReduced In-Sample shape: {df_insample_reduced.shape}")
print(f"Reduced Out-of-Sample shape: {df_outsample_reduced.shape}")
print(f"\nReduced features: {df_insample_reduced.shape[1] - 1} (down from {X_insample.shape[1]})")


Reduced In-Sample shape: (1195, 17)
Reduced Out-of-Sample shape: (339, 17)

Reduced features: 16 (down from 25)


## Re-fit OLS with Reduced Variables

In [11]:
# Prepare data for OLS
y_insample_reduced = df_insample_reduced['cl1_ret']
X_insample_reduced = df_insample_reduced.drop('cl1_ret', axis=1)

y_outsample_reduced = df_outsample_reduced['cl1_ret']
X_outsample_reduced = df_outsample_reduced.drop('cl1_ret', axis=1)

# Add constant
X_insample_reduced_const = sm.add_constant(X_insample_reduced)
X_outsample_reduced_const = sm.add_constant(X_outsample_reduced)

# Fit OLS on reduced variables
ols_reduced = sm.OLS(y_insample_reduced, X_insample_reduced_const).fit()

print(ols_reduced.summary())

                            OLS Regression Results                            
Dep. Variable:                cl1_ret   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.050
Method:                 Least Squares   F-statistic:                     4.953
Date:                Mon, 17 Nov 2025   Prob (F-statistic):           5.41e-10
Time:                        17:49:11   Log-Likelihood:                 1915.3
No. Observations:                1195   AIC:                            -3797.
Df Residuals:                    1178   BIC:                            -3710.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

## Reduced OLS Performance

In [12]:
# Make predictions
y_pred_reduced_insample = ols_reduced.predict(X_insample_reduced_const)
y_pred_reduced_outsample = ols_reduced.predict(X_outsample_reduced_const)

# Calculate metrics
ols_red_r2_insample = r2_score(y_insample_reduced, y_pred_reduced_insample)
ols_red_rmse_insample = np.sqrt(mean_squared_error(y_insample_reduced, y_pred_reduced_insample))
ols_red_mae_insample = mean_absolute_error(y_insample_reduced, y_pred_reduced_insample)

ols_red_r2_outsample = r2_score(y_outsample_reduced, y_pred_reduced_outsample)
ols_red_rmse_outsample = np.sqrt(mean_squared_error(y_outsample_reduced, y_pred_reduced_outsample))
ols_red_mae_outsample = mean_absolute_error(y_outsample_reduced, y_pred_reduced_outsample)

print("="*70)
print("OLS WITH REDUCED VARIABLES (Lasso-selected) PERFORMANCE")
print("="*70)
print("\nIn-Sample:")
print(f"  R² Score:  {ols_red_r2_insample:.6f}")
print(f"  RMSE:      {ols_red_rmse_insample:.6f}")
print(f"  MAE:       {ols_red_mae_insample:.6f}")

print("\nOut-of-Sample:")
print(f"  R² Score:  {ols_red_r2_outsample:.6f}")
print(f"  RMSE:      {ols_red_rmse_outsample:.6f}")
print(f"  MAE:       {ols_red_mae_outsample:.6f}")

print("\nModel Diagnostics:")
print(f"  Adjusted R²: {ols_reduced.rsquared_adj:.6f}")
print(f"  AIC: {ols_reduced.aic:.2f}")
print(f"  BIC: {ols_reduced.bic:.2f}")
print(f"  F-statistic: {ols_reduced.fvalue:.4f}")
print(f"  Prob (F-stat): {ols_reduced.f_pvalue:.2e}")

OLS WITH REDUCED VARIABLES (Lasso-selected) PERFORMANCE

In-Sample:
  R² Score:  0.063033
  RMSE:      0.048718
  MAE:       0.036127

Out-of-Sample:
  R² Score:  -0.008185
  RMSE:      0.061935
  MAE:       0.042616

Model Diagnostics:
  Adjusted R²: 0.050307
  AIC: -3796.61
  BIC: -3710.15
  F-statistic: 4.9530
  Prob (F-stat): 5.41e-10


## Reduced OLS Coefficients

In [13]:
# Create coefficient summary
coef_reduced_summary = pd.DataFrame({
    'Coefficient': ols_reduced.params,
    'Std Error': ols_reduced.bse,
    't-statistic': ols_reduced.tvalues,
    'p-value': ols_reduced.pvalues,
    '95% CI Lower': ols_reduced.conf_int()[0],
    '95% CI Upper': ols_reduced.conf_int()[1]
})

print("\nReduced OLS Regression Coefficients:")
print(coef_reduced_summary.round(6))

# Count significant variables
sig_count = (ols_reduced.pvalues < 0.05).sum()
print(f"\nSignificant variables (p < 0.05): {sig_count} out of {len(ols_reduced.params)}")


Reduced OLS Regression Coefficients:
                                   Coefficient  Std Error  t-statistic  \
const                                 0.001332   0.001432     0.930637   
dow_jones_us_oilandgas_index_ret      0.508707   0.130498     3.898185   
exxonmobil_ret                       -0.433593   0.104623    -4.144327   
weekly_commercial_short_diff          0.000000   0.000000     4.902879   
weekly_commercial_long_diff          -0.000000   0.000000    -4.386068   
weekly_change_in_crude_stock_diff    -0.000001   0.000000    -3.637183   
cl2_ret                              -0.115889   0.034223    -3.386298   
cl1_ret_l2                           -0.083933   0.029043    -2.889943   
spx_ret                              -0.230574   0.107407    -2.146729   
cl1_ret_l4                            0.053144   0.029046     1.829629   
chevron_ret                          -0.161522   0.101017    -1.598952   
vix_diff                             -0.001392   0.000984    -1.414891   


## Model Comparison Summary

In [14]:
# Comparison table
comparison = pd.DataFrame({
    'Metric': ['R² (In-Sample)', 'RMSE (In-Sample)', 'MAE (In-Sample)',
               'R² (Out-of-Sample)', 'RMSE (Out-of-Sample)', 'MAE (Out-of-Sample)',
               'Number of Features'],
    'Lasso': [lasso_r2_insample, lasso_rmse_insample, lasso_mae_insample,
              lasso_r2_outsample, lasso_rmse_outsample, lasso_mae_outsample,
              X_insample.shape[1]],
    'OLS (Reduced)': [ols_red_r2_insample, ols_red_rmse_insample, ols_red_mae_insample,
                      ols_red_r2_outsample, ols_red_rmse_outsample, ols_red_mae_outsample,
                      len(selected_feature_names)]
})

print("\n" + "="*70)
print("MODEL COMPARISON: LASSO vs OLS (REDUCED)")
print("="*70)
print(comparison.to_string(index=False))


MODEL COMPARISON: LASSO vs OLS (REDUCED)
              Metric     Lasso  OLS (Reduced)
      R² (In-Sample)  0.058995       0.063033
    RMSE (In-Sample)  0.048823       0.048718
     MAE (In-Sample)  0.036032       0.036127
  R² (Out-of-Sample)  0.001513      -0.008185
RMSE (Out-of-Sample)  0.061636       0.061935
 MAE (Out-of-Sample)  0.042430       0.042616
  Number of Features 25.000000      16.000000


## Residual Analysis (Reduced OLS)

In [ ]:
# Calculate residuals
residuals_reduced_insample = y_insample_reduced - y_pred_reduced_insample
residuals_reduced_outsample = y_outsample_reduced - y_pred_reduced_outsample

print("In-Sample Residuals (Reduced OLS):")
print(f"  Mean: {residuals_reduced_insample.mean():.6f}")
print(f"  Std Dev: {residuals_reduced_insample.std():.6f}")
print(f"  Min: {residuals_reduced_insample.min():.6f}")
print(f"  Max: {residuals_reduced_insample.max():.6f}")

print("\nOut-of-Sample Residuals (Reduced OLS):")
print(f"  Mean: {residuals_reduced_outsample.mean():.6f}")
print(f"  Std Dev: {residuals_reduced_outsample.std():.6f}")
print(f"  Min: {residuals_reduced_outsample.min():.6f}")
print(f"  Max: {residuals_reduced_outsample.max():.6f}")

## Summary

**Process**:
1. Applied Lasso regression with cross-validated optimal alpha
2. Identified non-zero coefficients to select important variables
3. Reduced feature set from 25 features to selected variables
4. Re-fit OLS with reduced variables

**Key Results**:
- Lasso eliminated multicollinearity by shrinking less important variables to zero
- Reduced OLS has fewer features but maintains predictive power
- Comparison of in-sample and out-of-sample performance shows generalization ability